# 02_Feature_Engineering — Amazon Delivery Dataset

**Input:** `amazon_delivery.csv` (raw, 43,739 rows) — Weather and Traffic are real recorded
columns in this dataset, not simulated, so there's no leakage/defensibility concern here the way
there was with the earlier synthetic e-commerce dataset.

**This notebook builds:**
- `Distance` — haversine distance between store and drop location
- Row cleanup — removes bad coordinate rows and unrealistic distance outliers
- `Preparation_Time` — minutes between order placed and pickup
- `Order_Hour`, `Time_of_Day`, `Peak_Hour`, `Is_Weekend` — order-time features
- Missing-value handling

**Output:** `feature_engineered_delivery.csv` — a new file. The raw `amazon_delivery.csv` is
never modified.

All of the logic below is pulled from your EDA notebook (where it was originally written
inline) and reorganized here as a standalone, documented feature-engineering stage.


## Step 1 — Load Raw Dataset

In [18]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/amazon_delivery.csv"                   # original, untouched
OUT_PATH = "../data/processed/delivery_features_v2.csv"         # new file, written at the end

df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()


Shape: (43739, 16)


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


## Step 2 — Create `Distance`

Store and drop-off coordinates are given as latitude/longitude pairs, but there's no direct
distance column. The haversine formula converts the coordinate pair into a great-circle
distance in kilometers, which is a much more useful predictor than raw lat/lon values.


In [19]:
# Haversine distance between store and drop location, in kilometers.
lat1 = np.radians(df["Store_Latitude"])
lon1 = np.radians(df["Store_Longitude"])
lat2 = np.radians(df["Drop_Latitude"])
lon2 = np.radians(df["Drop_Longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)
c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df["Distance"] = 6371 * c  # Earth radius in km
df[["Store_Latitude", "Store_Longitude", "Drop_Latitude", "Drop_Longitude", "Distance"]].head()


,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Distance
0,22.745049,75.892471,22.765049,75.912471,3.025149
1,12.913041,77.683237,13.043041,77.813237,20.183530
2,12.914264,77.678400,12.924264,77.688400,1.552758
3,11.003669,76.976494,11.053669,77.026494,7.790401
4,12.972793,80.249982,13.012793,80.289982,6.210138


## Step 3 — Remove Bad Rows

Two data-quality issues showed up during EDA:

1. **Negative coordinates.** All deliveries in this dataset are within India, where latitude
   and longitude should both be positive. A negative value indicates a bad GPS read, not a real
   location.
2. **Unrealistic distances.** A small number of rows have a `Distance` far beyond what's
   plausible for a single delivery (a symptom of the same bad-coordinate issue) — these are
   dropped using a 100 km cutoff, based on what the EDA distance distribution showed.


In [20]:
# Flag rows with an impossible (negative) coordinate
bad_coords = (
    (df["Store_Latitude"] < 0) |
    (df["Store_Longitude"] < 0) |
    (df["Drop_Latitude"] < 0) |
    (df["Drop_Longitude"] < 0)
)
print("Rows with negative coordinates:", bad_coords.sum(),
      f"({bad_coords.mean()*100:.2f}% of data)")

# Drop them, then drop unrealistic distance outliers (> 100 km for a single delivery)
before = len(df)
df = df[~bad_coords]
df = df[df["Distance"] <= 100]
print(f"Rows removed: {before - len(df)} | Remaining: {len(df)}")

df["Distance"].describe()


Rows with negative coordinates: 188 (0.43% of data)
Rows removed: 188 | Remaining: 43551


count    43551.000000
mean         9.733995
std          5.604430
min          1.465067
25%          4.663412
50%          9.220209
75%         13.681416
max         20.969489
Name: Distance, dtype: float64

## Step 4 — Parse Order/Pickup Times

`Order_Time` and `Pickup_Time` are stored as text (`HH:MM:SS`). They need to become real time
values before any time-based feature (preparation time, order hour, etc.) can be computed.
A small number of rows have malformed time strings — those fail to parse and are handled in
Step 6 (missing values).


In [21]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d")

df["Order_Time"] = pd.to_datetime(df["Order_Time"], format="%H:%M:%S", errors="coerce")
df["Pickup_Time"] = pd.to_datetime(df["Pickup_Time"], format="%H:%M:%S", errors="coerce")

print("Unparseable Order_Time :", df["Order_Time"].isna().sum())
print("Unparseable Pickup_Time:", df["Pickup_Time"].isna().sum())


Unparseable Order_Time : 59
Unparseable Pickup_Time: 0


## Step 5 — Create `Preparation_Time`

Minutes between the order being placed and the order being picked up — a genuine operational
signal (a slow pickup is a real driver of overall delivery time) and, importantly, something
known *before* the delivery is complete, so it's safe to use as a predictive feature.

Because `Order_Time`/`Pickup_Time` only carry a time-of-day (no date), a pickup logged just
after midnight for an order placed just before midnight would otherwise compute as a large
*negative* duration. That's corrected by adding 24 hours whenever the raw difference is
negative.


In [22]:
# Drop rows where either time failed to parse (can't compute a duration without both)
df = df.dropna(subset=["Order_Time", "Pickup_Time"])

df["Preparation_Time"] = (df["Pickup_Time"] - df["Order_Time"]).dt.total_seconds() / 60

# Correct for pickups that roll over past midnight relative to the order time
df.loc[df["Preparation_Time"] < 0, "Preparation_Time"] += 24 * 60

df["Preparation_Time"].describe()


count    43492.000000
mean         9.992068
std          4.086908
min          5.000000
25%          5.000000
50%         10.000000
75%         15.000000
max         15.000000
Name: Preparation_Time, dtype: float64

## Step 6 — Create `Order_Hour`, `Time_of_Day`, `Peak_Hour`, `Is_Weekend`

Four features derived from `Order_Time` / `Order_Date`, capturing *when* the order was placed:

- `Order_Hour` — the raw hour (0–23), useful as a numeric feature
- `Time_of_Day` — a human-readable bucket (Morning / Afternoon / Evening / Night)
- `Peak_Hour` — flags typical rush-hour ordering windows (7–10 AM, 6–9 PM), which plausibly
  correlate with higher traffic and slower deliveries
- `Is_Weekend` — Saturday/Sunday orders may behave differently (different traffic patterns,
  staffing)


In [23]:
df["Order_Hour"] = df["Order_Time"].dt.hour

def time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["Time_of_Day"] = df["Order_Hour"].apply(time_of_day)

def peak_hour(hour):
    # Typical rush-hour windows: morning commute and evening commute/dinner rush
    if (7 <= hour <= 10) or (18 <= hour <= 21):
        return 1
    else:
        return 0

df["Peak_Hour"] = df["Order_Hour"].apply(peak_hour)

df["Is_Weekend"] = df["Order_Date"].dt.dayofweek >= 5

df[["Order_Hour", "Time_of_Day", "Peak_Hour", "Is_Weekend"]].head(10)


,Order_Hour,Time_of_Day,Peak_Hour,Is_Weekend
0,11,Morning,0,True
1,19,Evening,1,False
2,8,Morning,1,True
3,18,Evening,1,False
4,13,Afternoon,0,True
5,21,Night,1,False
6,19,Evening,1,False
7,17,Evening,0,False
8,20,Evening,1,True
9,21,Night,1,True


## Step 7 — Handle Missing Values

Three sources of missing data need handling:

- `Agent_Rating` — 54 missing values in the raw data (proper `NaN`, caught by `isnull()`)
- `Weather` — 91 missing values in the raw data (proper `NaN`, caught by `isnull()`)
- `Traffic` — 91 rows where the value is the **literal string `"NaN "`** (with a trailing
  space), not a real null. `isnull()` reports zero missing values for `Traffic` because of this
  — the string doesn't match pandas' default null detection. Left unhandled, these rows would
  survive as a meaningless fifth traffic category instead of being dropped like the genuine
  nulls.

All three are dropped rather than imputed — inventing a plausible weather/traffic/rating value
would reintroduce the kind of "simulated-but-treated-as-real" problem worth avoiding. Dropping
this small a slice of the data is a low-cost, defensible choice.


In [24]:
print("Missing values before cleanup (isnull only):\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\n'Traffic' rows stored as the literal string 'NaN ' (missed by isnull()):",
      (df["Traffic"] == "NaN ").sum())

# Treat the disguised "NaN " string as missing before dropping
df["Traffic"] = df["Traffic"].replace("NaN ", np.nan)

df = df.dropna(subset=["Agent_Rating", "Weather", "Traffic"])

print("\nMissing values after cleanup:\n", df.isnull().sum().sum(), "total remaining")
print("Final shape:", df.shape)


Missing values before cleanup (isnull only):
 Agent_Rating    49
dtype: int64

'Traffic' rows stored as the literal string 'NaN ' (missed by isnull()): 0

Missing values after cleanup:
 0 total remaining
Final shape: (43443, 22)


## Step 8 — Clean Whitespace in Categorical Columns

EDA flagged this: `Traffic`, `Vehicle`, and `Area` carry a trailing space on every value
(e.g. `"Low "`, `"motorcycle "`, `"Urban "`). It's consistent within each column, so it wasn't
silently creating duplicate categories — but left in, it's a common source of subtle bugs later
(e.g. a dictionary lookup or `.map()` keyed on `"Low"` silently failing against `"Low "`, or a
one-hot encoder producing categories no one intended to create). Stripped here, once, so every
downstream step works with clean values.

Done *after* the missing-value step above — the disguised `"NaN "` match there depends on the
exact untrimmed string, so stripping first would have broken that check.


In [25]:
# Strip trailing/leading whitespace from all categorical columns used as features.
categorical_cols = ["Weather", "Traffic", "Vehicle", "Area", "Category"]
for col in categorical_cols:
    df[col] = df[col].str.strip()

# Confirm: no more stray whitespace in any category label
for col in categorical_cols:
    print(col, "->", sorted(df[col].unique()))


Weather -> ['Cloudy', 'Fog', 'Sandstorms', 'Stormy', 'Sunny', 'Windy']
Traffic -> ['High', 'Jam', 'Low', 'Medium']
Vehicle -> ['motorcycle', 'scooter', 'van']
Area -> ['Metropolitian', 'Other', 'Semi-Urban', 'Urban']
Category -> ['Apparel', 'Books', 'Clothing', 'Cosmetics', 'Electronics', 'Grocery', 'Home', 'Jewelry', 'Kitchen', 'Outdoors', 'Pet Supplies', 'Shoes', 'Skincare', 'Snacks', 'Sports', 'Toys']


## Step 9 — Flag Quick-Commerce Orders (`Is_Quick_Commerce`)

EDA turned up something worth acting on: `Grocery` orders (6.2% of the data) average 26.5
minutes with a std of 9.5 — a tight, fast, low-variance cluster completely unlike every other
category (~130 min, std ~47). That's the signature of a quick-commerce delivery model mixed
into an otherwise standard-delivery dataset, not just a fast category.

**Decision:** keep `Grocery` in the same dataset and the same model, rather than training two
separate models. `Category` is already a feature, and a tree-based model (Decision Tree /
Random Forest) will naturally split on it and learn the different behavior without losing any
data or adding a second pipeline. A boolean flag is added anyway, purely so the distinction is
explicit in the data/documentation rather than implicit inside one-hot encoding — and so a
simpler baseline model (e.g. Linear Regression) has direct access to it too.


In [26]:
# Is_Quick_Commerce Rule
# Grocery orders behave like a different delivery model entirely (tight, fast,
# low-variance) compared to every other category. Flagged explicitly rather than
# left implicit inside the one-hot-encoded Category column.
df["Is_Quick_Commerce"] = df["Category"] == "Grocery"

df.groupby("Is_Quick_Commerce")["Delivery_Time"].agg(["mean", "std", "count"]).round(1)


,mean,std,count
Is_Quick_Commerce,,,
False,131.4,46.8,40765
True,26.5,9.5,2678


## Step 10 — Save Engineered Dataset

The raw file is left untouched; the engineered dataframe is written to a new CSV.


In [27]:
df.to_csv(OUT_PATH, index=False)
print("Raw file (untouched):", RAW_PATH)
print("Engineered file (new):", OUT_PATH, "| shape:", df.shape)


Raw file (untouched): ../data/raw/amazon_delivery.csv
Engineered file (new): ../data/processed/delivery_features_v2.csv | shape: (43443, 23)


## Step 11 — Feature Set Preview

A quick preview of the columns available for the ML stage — raw categorical/numeric columns
plus everything engineered above. This is descriptive only; building `X`/`y` and training
belongs in the next notebook (`03_Model_Training.ipynb`), not here.


In [28]:
feature_preview = [
    "Agent_Age", "Agent_Rating", "Weather", "Traffic", "Vehicle", "Area", "Category",
    "Distance", "Preparation_Time", "Order_Hour", "Time_of_Day", "Peak_Hour", "Is_Weekend",
]
df[feature_preview + ["Delivery_Time"]].head()


,Agent_Age,Agent_Rating,Weather,Traffic,Vehicle,Area,Category,Distance,Preparation_Time,Order_Hour,Time_of_Day,Peak_Hour,Is_Weekend,Delivery_Time
0,37,4.9,Sunny,High,motorcycle,Urban,Clothing,3.025149,15.0,11,Morning,0,True,120
1,34,4.5,Stormy,Jam,scooter,Metropolitian,Electronics,20.183530,5.0,19,Evening,1,False,165
2,23,4.4,Sandstorms,Low,motorcycle,Urban,Sports,1.552758,15.0,8,Morning,1,True,130
3,38,4.7,Sunny,Medium,motorcycle,Metropolitian,Cosmetics,7.790401,10.0,18,Evening,1,False,105
4,32,4.6,Cloudy,High,scooter,Metropolitian,Toys,6.210138,15.0,13,Afternoon,0,True,150
